# Phase 2A — Forum threaded → summary dengan OpenAI API
Notebook ini mensimulasikan satu thread forum: laporan awal, beberapa balasan penumpang, dan laporan kondisi parah. Setiap pesan diproses berurutan dengan `previous_summary`, sehingga state forum terus diperbarui. Output tiap pemanggilan dibatasi `160` token dan klaim penumpang tidak dapat menyelesaikan insiden tanpa konfirmasi petugas.

`RUN_LIVE_OPENAI=True` menjalankan satu pemanggilan OpenAI untuk setiap pesan forum. Untuk pengujian tanpa biaya, ubah ke `False` terlebih dahulu.

In [ ]:
from pathlib import Path
import sys, json, tempfile
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'backend' / 'jakroute').is_dir()), None)
assert ROOT, 'Buka notebook dari folder paket yang sudah diekstrak lengkap.'
sys.path.insert(0, str(ROOT / 'backend'))
DATA = ROOT / 'backend' / 'data'
from jakroute.providers import load_json
from jakroute.config import Settings

from dataclasses import replace
from jakroute.forum_state import ForumStore,OpenAISummarizer,DemoSummarizer
RUN_LIVE_OPENAI=False
settings=replace(Settings.from_env(),forum_mode='openai')
if RUN_LIVE_OPENAI:
 assert settings.openai_api_key, 'Isi OPENAI_API_KEY di backend/.env terlebih dahulu.'
processor=OpenAISummarizer(settings) if RUN_LIVE_OPENAI else DemoSummarizer()
work=tempfile.TemporaryDirectory(prefix='jakroute_forum_openai_')
store=ForumStore(Path(work.name)/'forum.sqlite3')

# Satu thread utama: setiap item setelah item pertama adalah balasan.
# Metadata thread/parent hanya untuk simulasi forum; ForumStore tetap menyimpan
# seluruh report mentah dan memakai summary terstruktur sebagai state routing.
FORUM_THREAD=[
 {'report_id':'thread_001_post','thread_id':'thread_escalator','author':'Rina','resource_id':'escalator_link','observed_at':'2026-09-07T08:00:00Z','message':'Eskalator menuju peron 1 rusak. Penumpang harus lewat tangga.'},
 {'report_id':'thread_001_reply_01','thread_id':'thread_escalator','parent_report_id':'thread_001_post','author':'Dimas','resource_id':'escalator_link','observed_at':'2026-09-07T08:03:00Z','message':'Saya baru lewat. Benar, eskalator masih tidak bergerak dan tangga menjadi satu-satunya akses ke peron 1.'},
 {'report_id':'thread_001_reply_02','thread_id':'thread_escalator','parent_report_id':'thread_001_reply_01','author':'Sari','resource_id':'escalator_link','observed_at':'2026-09-07T08:07:00Z','message':'Tangga cukup padat, tetapi eskalatornya masih rusak. Belum terlihat petugas memperbaiki.'},
 {'report_id':'thread_001_reply_03','thread_id':'thread_escalator','parent_report_id':'thread_001_reply_02','author':'Bimo','resource_id':'escalator_link','observed_at':'2026-09-07T08:12:00Z','message':'Sepertinya sudah diperbaiki, saya melihat lampunya menyala. Mohon petugas mengecek karena saya belum mendapat konfirmasi resmi.'},
 {'report_id':'thread_001_reply_04','thread_id':'thread_escalator','parent_report_id':'thread_001_reply_03','author':'Petugas lapangan','resource_id':'escalator_link','observed_at':'2026-09-07T08:20:00Z','message':'Saya cek dari bawah, eskalator masih berhenti dan penumpang tetap harus menggunakan tangga. Belum ada konfirmasi perbaikan.'},
]

# Thread kedua: contoh kondisi parah yang harus memblokir routing.
SEVERE_REPORT={'report_id':'thread_002_post','thread_id':'thread_fire','author':'Ayu','resource_id':'north','observed_at':'2026-09-07T08:25:00Z','message':'Ada asap tebal dan api terlihat di koridor utara. Jalur harus ditutup.'}

print((f'OPENAI LIVE — {len(FORUM_THREAD)+1} pemanggilan, max_output_tokens=160 each' if RUN_LIVE_OPENAI else 'FIXTURE SIMULATION — OpenAI belum dipanggil'))

In [ ]:
evaluation=[]
for index, report in enumerate(FORUM_THREAD, start=1):
    previous=store.snapshot()
    summary=processor.summarize(report,previous)
    changed=store.update(report,summary)
    current_state=store.snapshot()
    item={'sequence':index,'author':report['author'],'report_id':report['report_id'],
          'parent_report_id':report.get('parent_report_id'),'message':report['message'],
          'summary':summary,'changed':changed,'state_version':current_state['version'],
          'model_reply':current_state['summary']}
    evaluation.append(item)
    print(f"{report['author']}: {report['message']}")
    print(f"model: {current_state['summary']}")
    print()

print('\nSTATE SETELAH THREAD ESKALATOR:')
print(json.dumps(store.snapshot(),ensure_ascii=False,indent=2))

In [ ]:
active=store.snapshot()['incidents']
assert len(evaluation)==5
assert evaluation[-1]['state_version'] >= 1
assert any(i['resource_id']=='escalator_link' for i in active)
# Balasan penumpang yang mengira sudah diperbaiki tidak langsung menghapus insiden.
assert any(i['resource_id']=='escalator_link' and i['status']!='resolved' for i in active)
print('PASS: balasan thread diproses berurutan; insiden belum resolved tanpa konfirmasi petugas.')

In [ ]:
previous=store.snapshot()
severe_summary=processor.summarize(SEVERE_REPORT,previous)
store.update(SEVERE_REPORT,severe_summary)
severe_incident=next(i for i in store.snapshot()['incidents'] if i['resource_id']=='north')
print('\nTHREAD KONDISI PARAH:')
print(json.dumps({'report':SEVERE_REPORT,'summary':severe_summary,'incident':severe_incident},ensure_ascii=False,indent=2))
assert severe_incident['routing_code']==-1
assert severe_incident['status']!='resolved'
print('PASS: kondisi parah menghasilkan routing_code=-1 dan tetap aktif.')

## Catatan pengujian
Mode live melakukan `len(FORUM_THREAD)+1` request OpenAI. Setiap request memakai summary state sebelumnya, bukan mengirim seluruh percakapan tanpa batas. Untuk membuka insiden, tetap gunakan `store.confirm(...)` dengan otoritas petugas.

In [ ]:
work.cleanup()